# Lab 18 — MITRE ATT&CK Threat Mapping with Python

*AI Cybersecurity Masterclass · Lab 18 · AI60secLab*

**Research question:** How can machine-readable MITRE ATT&CK data be transformed into an analyzable threat-technique dataset?

This notebook downloads the official Enterprise ATT&CK STIX 2.1 bundle at runtime. If network access is unavailable, place `enterprise-attack.json` in `data/raw/`.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
from src.mitre_data import (
    ATTACK_ENTERPRISE_URL,
    load_remote_or_local,
    technique_catalog,
    object_catalog,
    relationship_table,
)

local_fallback = DATA_DIR / "enterprise-attack.json"

attack_bundle = load_remote_or_local(
    ATTACK_ENTERPRISE_URL,
    local_path=local_fallback,
)

techniques = technique_catalog(
    attack_bundle,
    id_prefix="T",
)
groups = object_catalog(
    attack_bundle,
    "intrusion-set",
)
relationships = relationship_table(
    attack_bundle,
)

print("Active techniques/sub-techniques:", len(techniques))
print("Active groups:", len(groups))
print("Active relationships:", len(relationships))

display(techniques.head())

In [ ]:
tactic_rows = techniques.copy()
tactic_rows["tactic"] = tactic_rows["tactics"].str.split(", ")
tactic_rows = tactic_rows.explode("tactic")
tactic_rows = tactic_rows[
    tactic_rows["tactic"].notna()
    & (tactic_rows["tactic"] != "")
]

tactic_counts = (
    tactic_rows.groupby("tactic", as_index=False)
    .agg(technique_count=("external_id", "nunique"))
    .sort_values("technique_count", ascending=False)
)

display(tactic_counts)

In [ ]:
query = "credential"

mask = (
    techniques["name"].str.contains(
        query, case=False, na=False
    )
    | techniques["description"].str.contains(
        query, case=False, na=False
    )
)

display(
    techniques.loc[
        mask,
        ["external_id", "name", "tactics"]
    ].head(30)
)

## Research task

Choose one tactic and document:
- the techniques mapped to it;
- which behaviors your available telemetry could observe;
- which behaviors require endpoint, identity, cloud, or application telemetry.

**Important:** ATT&CK organizes observed adversary behavior. A network-flow classifier does not automatically produce ATT&CK attribution.